# Полный программный запуск проекта

Этот ноутбук объединяет аудит, retrieval-эксперимент, финальную демонстрационную оценку и результаты HTTP-проверок. Статьи описывают сам проект. Корпус и вопросы составлены автоматически; это не банковская база и не независимая экспертная оценка. Все изменения подробно объяснены в docs/development_guide.docx.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data' / 'demo'
display(json.loads((DATA / 'provenance.json').read_text(encoding='utf-8')))


## 1. Проверяем статьи и независимость файлов


In [ ]:
from src.rag.audit import audit_corpus
from src.rag.evaluation import load_queries, require_independent_query_splits
display(audit_corpus(DATA / 'articles.jsonl'))
display(require_independent_query_splits(load_queries(DATA / 'calibration.jsonl'), load_queries(DATA / 'holdout.jsonl')))
articles = [json.loads(line) for line in (DATA / 'articles.jsonl').read_text(encoding='utf-8').splitlines()]
display(pd.DataFrame(articles)[['id', 'title', 'language', 'owner', 'data_role']])


## 2. Сравниваем методы и смотрим фиксированный результат

Выбор производится только по calibration. Результаты предыдущего запуска с такими же файлами и кодом читаются из кэша. Не подбирайте настройки по демонстрационному holdout: при необходимости настоящей оценки нужен новый экспертный набор.


In [ ]:
from scripts.run_retrieval_experiment import run
results, run_dir = run(DATA, ROOT / 'work' / 'demo_runs')
display(pd.DataFrame(results['candidate_metrics']))
print('Выбран:', results['selected_method'], results['selected_parameters'])
display(pd.DataFrame(results['holdout_threshold_metrics']))
display(pd.DataFrame.from_dict(results['holdout_ranking']['language_metrics'], orient='index'))
display(results['operational'])
display(pd.read_csv(run_dir / 'calibration_cases.csv'))


## 3. Проверяем весь HTTP-путь

Следующая ячейка запускает временный сервер только на loopback, проверяет реальные HTTP-запросы и затем останавливает сервер. Нужен доступ к локальным сокетам. Для классификации требуется сохранённая модель из ноутбука 01. Проверка не повторяет official test BANKING77.


In [ ]:
import subprocess
completed = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'verify_demo_http.py')], cwd=ROOT, capture_output=True, text=True, timeout=90)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError('HTTP-проверка не прошла')


## 4. Самостоятельная работа с API

В терминале PyCharm из корня запустите `python scripts/start_demo.py`. Swagger: http://127.0.0.1:8000/docs. Поиск: POST /v1/knowledge/search. Ответ с цитатой: POST /v1/knowledge/answer, поля text, language и mode=extractive. Неподдержанный запрос приводит к abstained. Режим ollama требует отдельно запущенного сервера и OLLAMA_MODEL; контракт провайдера проверен тестами, а локальная Qwen2.5 0.5B реально вызвана на четырёх демонстрационных вопросах. Для совместного запуска API и LLM используйте compose.llm.yaml; подробности и ограничения приведены в Word.

При настроенном SUPPORT_API_KEY передавайте x-api-key. Endpoint /metrics показывает счётчики одного процесса. Внешний доступ требует TLS и централизованных ограничений. Команды Docker и разбор архитектуры находятся в Word.


In [ ]:
example_payload = {'text': 'Как запустить локальный API?', 'language': 'ru', 'mode': 'extractive'}
print(json.dumps(example_payload, ensure_ascii=False, indent=2))
